# SQGDP2
- go into folder `SQGDP`
- geofip and state

In [5]:
import os

BEA_API_KEY = os.environ.get('BEA_API_KEY')

# get params available

In [6]:
from typing import Any
import requests


BEA_API_URL = "https://apps.bea.gov/api/data/"


def get_bea_geofips(
    api_key: str,
    table_name: str,
    line_code: str | int | None = None,
    timeout: float = 30.0,
) -> list[dict[str, str]]:
    """
    Return valid GeoFips parameters for a BEA Regional table.

    Parameters
    ----------
    api_key:
        Your BEA API key.
    table_name:
        Regional table name, such as "CAINC4" or "SAINC1".
    line_code:
        Optional line code used to further filter available geographies.
    timeout:
        HTTP request timeout in seconds.

    Returns
    -------
    list[dict[str, str]]
        Items in the form:
        [
            {"Key": "01000", "Desc": "Alabama"},
            {"Key": "02000", "Desc": "Alaska"},
            ...
        ]

    Raises
    ------
    ValueError:
        If BEA returns an API error or an unexpected response.
    requests.HTTPError:
        If the HTTP request fails.
    """
    params: dict[str, Any] = {
        "UserID": api_key,
        "Method": "GetParameterValuesFiltered",
        "DatasetName": "Regional",
        "TargetParameter": "GeoFips",
        "TableName": table_name,
        "ResultFormat": "JSON",
    }

    if line_code is not None:
        params["LineCode"] = str(line_code)

    response = requests.get(
        BEA_API_URL,
        params=params,
        timeout=timeout,
    )
    response.raise_for_status()

    payload = response.json()

    try:
        results = payload["BEAAPI"]["Results"]
    except (KeyError, TypeError) as exc:
        raise ValueError("Unexpected response from the BEA API.") from exc

    if "Error" in results:
        error = results["Error"]

        if isinstance(error, list):
            messages = [
                item.get("APIErrorDescription", str(item))
                for item in error
            ]
            message = "; ".join(messages)
        else:
            message = error.get("APIErrorDescription", str(error))

        raise ValueError(f"BEA API error: {message}")

    values = results.get("ParamValue", [])

    if isinstance(values, dict):
        values = [values]

    return [
        {
            "Key": str(item["Key"]),
            "Desc": str(item["Desc"]),
        }
        for item in values
    ]

In [9]:


geofips = get_bea_geofips(BEA_API_KEY, "SQGDP2")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)